### Importing Libraries


In [1]:
import os
import pickle
import numpy as np

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

from keras.utils import to_categorical
from music21 import converter, instrument, note, chord, stream

### Data Preparation

    <!-- https://bitmidi.com use for more songs -->


In [2]:
def get_notes():
    notes = []
    dir = './midi_songs'

    for file in os.listdir(dir):
        if file.endswith('.mid'):
            file_path = os.path.join(dir, file)
            print(f'Parsing {file_path}...')
            midi = converter.parse(file_path)
            notes_to_parse = None
            parts = instrument.partitionByInstrument(midi)
            if parts:
                notes_to_parse = parts.parts[0].recurse()
            else:
                notes_to_parse = midi.flat.notes

            for element in notes_to_parse:
                if isinstance(element, note.Note):
                    notes.append(str(element.pitch))
                elif isinstance(element, chord.Chord):
                    notes.append('.'.join(str(n) for n in element.normalOrder))

    with open('./data/notes', 'wb') as file:
        pickle.dump(notes, file)
    
    return notes

In [3]:
def prepare_sequences(notes, n_vocab):
    seq_length = 100

    pitchnames = sorted(set(item for item in notes))

    note_to_int = dict((note, number) for number, note in enumerate(pitchnames))

    network_input = []
    network_output = []

    for i in range(0, len(notes) - seq_length):
        sequence_in = notes[i : i + seq_length]
        sequence_out = notes[i + seq_length]
        network_input.append([note_to_int[char] for char in sequence_in])
        network_output.append(note_to_int[sequence_out])

    n_patterns = len(network_input)

    network_input = np.reshape(network_input, (n_patterns, seq_length, 1))

    network_input = network_input / float(n_vocab)

    # network_output = to_categorical(network_output)

    return (network_input, network_output)

### Create Model


In [4]:
class MusicDataset(Dataset):
    def __init__(self, network_input, network_output):
        self.network_input = torch.tensor(network_input, dtype=torch.float32)
        self.network_output = torch.tensor(network_output, dtype=torch.long)
    
    def __len__(self):
        return len(self.network_input)
    
    def __getitem__(self, idx):
        return self.network_input[idx], self.network_output[idx]

In [5]:
class MusicLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, n_vocab):
        super(MusicLSTM, self).__init__()
        self.lstm1 = nn.LSTM(input_size, hidden_size, batch_first=True, dropout=0.3)
        self.lstm2 = nn.LSTM(hidden_size, hidden_size, batch_first=True, dropout=0.3)
        self.lstm3 = nn.LSTM(hidden_size, hidden_size, batch_first=True)
        self.batch_norm1 = nn.BatchNorm1d(hidden_size)
        self.batch_norm2 = nn.BatchNorm1d(256)
        self.fc1 = nn.Linear(hidden_size, 256)
        self.fc2 = nn.Linear(256, n_vocab)
        self.relu = nn.ReLU()
        self.softmax = nn.Softmax(dim=1)
        self.dropout = nn.Dropout(0.3)
    
    def forward(self, x):
        x, _ = self.lstm1(x)
        x, _ = self.lstm2(x)
        x, (hn, _) = self.lstm3(x)
        x = hn[-1]
        x = self.batch_norm1(x)
        x = self.dropout(x)
        x = self.fc1(x)
        x = self.relu(x)
        x = self.batch_norm2(x)
        x = self.dropout(x)
        x = self.fc2(x)
        return x

### Training Model


In [6]:
def train_network():
    notes = get_notes()
    n_vocab = len(set(notes))
    network_input, network_output = prepare_sequences(notes, n_vocab)
    dataset = MusicDataset(network_input, network_output)
    train_loader = DataLoader(dataset, batch_size=128, shuffle=True)
    
    input_size = 1
    hidden_size = 512
    model = MusicLSTM(input_size, hidden_size, n_vocab)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.RMSprop(model.parameters(), lr=0.001)
    
    epochs = 200
    best_loss = float("inf")
    best_model_path = "best_model.pth"
    
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for inputs, targets in train_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets.view(-1))
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        
        epoch_loss = total_loss / len(train_loader)
        print(f"Epoch {epoch + 1}/{epochs}, Loss: {epoch_loss}")
        
        if epoch_loss < best_loss:
            best_loss = epoch_loss
            torch.save(model.state_dict(), best_model_path)
            print(f"New best model saved with loss {best_loss:.4f}")


### Getting Prediction


In [7]:
def generate_notes(model, network_input, pitchnames, n_vocab, device, sequence_length=100, num_notes=500):
    int_to_note = {number: note for number, note in enumerate(pitchnames)}

    start_idx = np.random.randint(0, len(network_input) - 1)
    pattern = network_input[start_idx]
    prediction_output = []

    for _ in range(num_notes):
        input_tensor = torch.tensor(pattern, dtype=torch.float32).view(1, -1, 1).to(device)
        prediction = model(input_tensor).cpu().detach().numpy()
        index = np.argmax(prediction)
        result = int_to_note[index]
        prediction_output.append(result)

        pattern = np.append(pattern[1:], [[index / float(n_vocab)]], axis=0)

    return prediction_output

In [8]:
def create_midi(prediction_output, output_file='test_output.mid'):
    offset = 0
    output_notes = []

    for pattern in prediction_output:
        if '.' in pattern or pattern.isdigit():
            notes_in_chord = pattern.split('.')
            notes = []
            for n in notes_in_chord:
                new_note = note.Note(int(n))
                new_note.storedInstrument = instrument.Piano()
                notes.append(new_note)
            new_chord = chord.Chord(notes)
            new_chord.offset = offset
            output_notes.append(new_chord)
        else:
            new_note = note.Note(pattern)
            new_note.offset = offset
            new_note.storedInstrument = instrument.Piano()
            output_notes.append(new_note)

        offset += 0.5

    midi_stream = stream.Stream(output_notes)
    midi_stream.write('midi', fp=output_file)

In [9]:
def prepare_sequences_prediction(notes, pitchnames, n_vocab):
    note_to_int = dict((note, number) for number, note in enumerate(pitchnames))

    sequence_length = 100
    network_input = []
    output = []
    for i in range(0, len(notes) - sequence_length, 1):
        sequence_in = notes[i:i + sequence_length]
        sequence_out = notes[i + sequence_length]
        network_input.append([note_to_int[char] for char in sequence_in])
        output.append(note_to_int[sequence_out])

    n_patterns = len(network_input)

    normalized_input = np.reshape(network_input, (n_patterns, sequence_length, 1))
    normalized_input = normalized_input / float(n_vocab)

    return (network_input, normalized_input)

In [10]:
def generate():
    with open('data/notes', 'rb') as filepath:
        notes = pickle.load(filepath)

    pitchnames = sorted(set(notes))
    n_vocab = len(pitchnames)
    network_input, normalized_input = prepare_sequences_prediction(notes, pitchnames, n_vocab)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = MusicLSTM(input_size=1, hidden_size=512, n_vocab=n_vocab).to(device)
    model.load_state_dict(torch.load('best_model.pth'))
    model.eval()

    prediction_output = generate_notes(model, normalized_input, pitchnames, n_vocab, device)
    create_midi(prediction_output)

In [13]:
def main():
    print("Hey, This is a music generator!")
    print("Enter your choice:")
    print("1. Train the model")
    print("2. Generate music")
    choice = int(input())
    if choice == 1:
        train_network()
    elif choice == 2:
        generate()
    else:
        print("Invalid choice")

In [14]:
if __name__ == "__main__":
    main()

Hey, This is a music generator!
Enter your choice:
1. Train the model
2. Generate music


C:\Users\amits\AppData\Local\Temp\ipykernel_20820\2547068351.py:11: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load('best_model.pth'))
